# Build the datasets

One-time conversion of vendor downloads into the normalized tables described in `docs/schema.md`. Each build streams the synapse file, writes pre- and post-sorted copies, derives edges and the type matrix, checks every invariant and records provenance in `manifest.json`. The same two calls are available as `connexplorer download` / `connexplorer build`.

In [ ]:
from pathlib import Path
import connexplorer as cnx
from connexplorer import ingest

DATA = Path("../../data")

## FlyWire FAFB v783

The Codex bundle (3 GB) holds every raw table. Building takes about 30 s and 11 GB of RAM.

In [ ]:
RAW = DATA / "flywire" / "raw"
if not (RAW / "neurons.csv.gz").exists():
    ingest.download_raw("flywire", RAW)
res = ingest.build(ingest.FlyWireSource(), DATA / "flywire_783", RAW)
res.report.counts

## Male CNS v1.0

Put `body-annotations`, `body-neurotransmitters` and `syn-partners` for the release (from `gs://flyem-male-cns/v1.0/connectome-data/flat-connectome/`) in the raw folder; `connectome-weights` is optional and only used by the check below. About 45 s and 20 GB of RAM.

In [ ]:
RAW = DATA / "mcns" / "raw"
res = ingest.build(ingest.McnsSource("v1.0"), DATA / "mcns_v1.0", RAW)
res.report.counts

In [ ]:
# optional: the edge table must equal the vendor weights file pair for pair
ingest.verify_against_weights(DATA / "mcns_v1.0" / "tables", RAW, "v1.0")

## Skeletons

Drop the SWC zip (FlyWire: `sk_lod1_783_healed.zip`) into `data/<dataset>/tables/skeletons/`, or point `cnx.config.skeletons["flywire"]` at it.

In [ ]:
for name, version, path, m in cnx.dataset.available():
    print(name, version, path, m.tables["cells"]["rows"], "cells")